# 05. CNN Data Preparation

This notebook prepares the multimodal dataset required for the deep-learning model.

The CNN will combine:

- raw Sentinel-2 multispectral image patches;
- ERA5-Land weather predictors;

to estimate the probability of wildfire occurrence during the following seven days.

The temporal experimental design remains unchanged:

- **Training:** 2017–2022
- **Validation:** 2023
- **Test:** 2024–2025

The test period remains untouched during model development.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet(
    "../data/processed/model_dataset_full.parquet"
)

print(df.shape)
print(df["reference_date"].min())
print(df["reference_date"].max())

(875238, 40)
2017-05-01 00:00:00
2025-10-20 00:00:00


In [3]:
df["split"] = np.select(
    [
        df["iso_year"].between(2017, 2022),
        df["iso_year"] == 2023,
        df["iso_year"].between(2024, 2025),
    ],
    [
        "train",
        "validation",
        "test",
    ],
    default="unknown"
)

df["split"].value_counts()

split
train         584766
test          191100
validation     99372
Name: count, dtype: int64

In [4]:
cnn_manifest_all = df[
    [
        "cell_id",
        "reference_date",
        "selected_date",
        "fire_next_7_days",
        "split"
    ]
].copy()

cnn_manifest_all["has_image"] = (
    cnn_manifest_all["selected_date"].notna()
)

cnn_manifest_all.head()

,cell_id,reference_date,selected_date,fire_next_7_days,split,has_image
0,PT_0001,2017-05-01,2017-04-28,0,train,True
1,PT_0001,2017-05-08,2017-05-05,0,train,True
2,PT_0001,2017-05-15,2017-05-08,0,train,True
3,PT_0001,2017-05-22,2017-05-18,0,train,True
4,PT_0001,2017-05-29,2017-05-25,0,train,True


In [5]:
# Each cell-week must be unique
assert not cnn_manifest_all.duplicated(
    ["cell_id", "reference_date"]
).any()

# Selected Sentinel-2 image must always be before the prediction week
rows_with_image = cnn_manifest_all["has_image"]

assert (
    cnn_manifest_all.loc[rows_with_image, "selected_date"]
    <
    cnn_manifest_all.loc[rows_with_image, "reference_date"]
).all()

print("Manifest checks passed.")

Manifest checks passed.


In [6]:
cnn_availability = (
    cnn_manifest_all
    .groupby("split")
    .agg(
        total_rows=("cell_id", "size"),
        rows_with_image=("has_image", "sum"),
        positives_all=("fire_next_7_days", "sum")
    )
)

cnn_availability["rows_without_image"] = (
    cnn_availability["total_rows"]
    - cnn_availability["rows_with_image"]
)

cnn_availability["image_availability_pct"] = (
    cnn_availability["rows_with_image"]
    / cnn_availability["total_rows"]
    * 100
)

cnn_availability

,total_rows,rows_with_image,positives_all,rows_without_image,image_availability_pct
split,,,,,
test,191100,190128,9225,972,99.491366
train,584766,572076,36919,12690,97.829901
validation,99372,97459,3996,1913,98.074910


In [7]:
image_availability_target = (
    cnn_manifest_all
    .groupby(["split", "has_image"])
    .agg(
        observations=("cell_id", "size"),
        positives=("fire_next_7_days", "sum"),
        positive_rate=("fire_next_7_days", "mean")
    )
)

image_availability_target["positive_rate_pct"] = (
    image_availability_target["positive_rate"] * 100
)

image_availability_target

observations  positives  positive_rate  \
split      has_image                                           
test       False               972          2       0.002058   
           True             190128       9223       0.048509   
train      False             12690        280       0.022065   
           True             572076      36639       0.064046   
validation False              1913         40       0.020910   
           True              97459       3956       0.040591   

                      positive_rate_pct  
split      has_image                     
test       False               0.205761  
           True                4.850943  
train      False               2.206462  
           True                6.404569  
validation False               2.090957  
           True                4.059143

In [8]:
cnn_manifest = (
    cnn_manifest_all[
        cnn_manifest_all["has_image"]
    ]
    .drop(columns="has_image")
    .reset_index(drop=True)
)

cnn_manifest.shape

(859663, 5)

### Sentinel-2 image availability

A total of **859,663 cell-week observations** have a valid Sentinel-2 acquisition
available before the corresponding prediction week.

Image availability is high across all temporal splits, ranging from approximately
97.8% to 99.5%.

However, observations without an available image show a lower wildfire prevalence
than observations with imagery. Satellite availability is therefore not completely
independent of the target distribution.

The CNN will necessarily be trained and evaluated only on observations with valid
imagery. Consequently, final comparisons between the CNN and tabular models will
also be performed on this common image-available subset to ensure a fair evaluation.

In [9]:
unique_patches = (
    cnn_manifest[
        ["cell_id", "selected_date"]
    ]
    .drop_duplicates()
)

print(f"CNN observations: {len(cnn_manifest):,}")
print(f"Unique Sentinel patches: {len(unique_patches):,}")

print(
    f"Average observations per patch: "
    f"{len(cnn_manifest) / len(unique_patches):.2f}"
)

CNN observations: 859,663
Unique Sentinel patches: 644,931
Average observations per patch: 1.33


In [10]:
patches_by_split = (
    cnn_manifest[
        ["cell_id", "selected_date", "split"]
    ]
    .drop_duplicates()
    .groupby("split")
    .size()
)

patches_by_split

split
test          149575
train         427997
validation     67359
dtype: int64